<a href="https://colab.research.google.com/github/Saransh-Basu-01/Advanced-Pytorch/blob/main/Debugging_and_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fixing a shape mismatch

In [2]:
import torch
import torch.nn as nn
class SimpleNN(nn.Module):
  def  __init__(self):
    super().__init__()
    self.layer1=nn.Linear(784,128)
    self.activation=nn.ReLU()
    self.layer2=nn.Linear(128,64)
    self.layer3=nn.Linear(100,10)

  def forward(self,x):
    x=self.layer1(x)
    x=self.activation(x)
    x=self.layer2(x)
    x=self.activation(x)
    x=self.layer3(x)
    return x

dummy_input = torch.randn(4, 784)
model = SimpleNN()

try:
    output = model(dummy_input)
    print("Model ran successfully!")
except RuntimeError as e:
    print(f"Caught an error: {e}")


Caught an error: mat1 and mat2 shapes cannot be multiplied (4x64 and 100x10)


# correcting Device placement

In [3]:
import torch
import torch.nn as nn

# Assume a CUDA-enabled GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)

    def forward(self, x):
        return self.linear(x)

# Create the model and move it to the target device (e.g., GPU)
model = SimpleNet().to(device)
print(f"Model parameters are on: {next(model.parameters()).device}")

# Create input data - intentionally left on the CPU
input_data = torch.randn(8, 10)
print(f"Input data is on: {input_data.device}")

# Attempt forward pass - this will likely cause an error if device is 'cuda'
try:
    output = model(input_data)
    print("Forward pass successful!")
except RuntimeError as e:
    print(f"\nCaught an error: {e}")
    print("\nHint: Check if the model and input data are on the same device.")



Using CPU
Model parameters are on: cpu
Input data is on: cpu
Forward pass successful!


# visualizing training with TensorBoard

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import time

# 1. Setup TensorBoard Writer
# Log files will be saved in the 'runs/simple_experiment' directory
writer = SummaryWriter('runs/simple_experiment')

# 2. Define a simple model, loss, and optimizer
model = nn.Linear(10, 2) # Simple linear model
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Simulate a simple dataset
inputs = torch.randn(100, 10) # 100 samples, 10 features
targets = torch.randn(100, 2) # 100 samples, 2 output values

# 3. Simple Training Loop
print("Starting simulated training...")
num_epochs = 50
for epoch in range(num_epochs):
    optimizer.zero_grad()    # Zero gradients
    outputs = model(inputs)  # Forward pass
    loss = criterion(outputs, targets) # Calculate loss

    # Simulate changing loss (replace with actual loss in real training)
    # Making loss decrease over epochs for demonstration
    simulated_loss = loss + torch.randn(1) * 0.1 + (num_epochs - epoch) / num_epochs

    simulated_loss.backward() # Backward pass (using simulated loss for demo)
    optimizer.step()       # Update weights

    # 4. Log metrics to TensorBoard
    if (epoch + 1) % 5 == 0: # Log every 5 epochs
        # Log the scalar 'loss' value
        writer.add_scalar('Training/Loss', simulated_loss.item(), epoch)

        # Log the distribution of model weights (example for the linear layer)
        writer.add_histogram('Model/Weights', model.weight, epoch)
        writer.add_histogram('Model/Bias', model.bias, epoch)

        print(f'Epoch [{epoch+1}/{num_epochs}], Simulated Loss: {simulated_loss.item():.4f}')

    time.sleep(0.1) # Simulate training time

# 5. Add model graph (optional)
# Ensure input shape matches what the model expects
# writer.add_graph(model, inputs[0].unsqueeze(0)) # Provide a sample input batch

# 6. Close the writer
writer.close()
print("Finished simulated training. TensorBoard logs saved to 'runs/simple_experiment'.")
print("Run 'tensorboard --logdir=runs' in your terminal to view.")





Starting simulated training...
Epoch [5/50], Simulated Loss: 2.2930
Epoch [10/50], Simulated Loss: 2.2329
Epoch [15/50], Simulated Loss: 1.9278
Epoch [20/50], Simulated Loss: 1.9508
Epoch [25/50], Simulated Loss: 1.8408
Epoch [30/50], Simulated Loss: 1.7673
Epoch [35/50], Simulated Loss: 1.5385
Epoch [40/50], Simulated Loss: 1.4559
Epoch [45/50], Simulated Loss: 1.2900
Epoch [50/50], Simulated Loss: 1.1593
Finished simulated training. TensorBoard logs saved to 'runs/simple_experiment'.
Run 'tensorboard --logdir=runs' in your terminal to view.


# Using Python Debugger

In [ ]:
import torch
import torch.nn as nn
import pdb # Import the debugger

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 128)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(128, 64)
        # Incorrect input size for layer3
        self.layer3 = nn.Linear(100, 10) # Error here!

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        x = self.activation(x)
        print("About to enter pdb...")
        pdb.set_trace() # Set a breakpoint here
        # Execution will pause here
        print("Shape before layer3:", x.shape) # We can inspect x in pdb
        x = self.layer3(x) # This line will cause an error
        return x

dummy_input = torch.randn(4, 784)
model = SimpleMLP()
output = model(dummy_input) # Run will now pause inside forward()

